24 hour simulation with dummy values for an IEEE 33 bus system

In [1]:
# imports
import pypsa
import numpy as np 
import pandas as pd 

Initialize the network and the snapshots, which will be taken on an hourly interval for a total of 24h

In [2]:
network = pypsa.Network()
snapshots = pd.date_range("2026-01-01 00:00", periods=24, freq="h")
network.set_snapshots(snapshots)
network.snapshots

DatetimeIndex(['2026-01-01 00:00:00', '2026-01-01 01:00:00',
               '2026-01-01 02:00:00', '2026-01-01 03:00:00',
               '2026-01-01 04:00:00', '2026-01-01 05:00:00',
               '2026-01-01 06:00:00', '2026-01-01 07:00:00',
               '2026-01-01 08:00:00', '2026-01-01 09:00:00',
               '2026-01-01 10:00:00', '2026-01-01 11:00:00',
               '2026-01-01 12:00:00', '2026-01-01 13:00:00',
               '2026-01-01 14:00:00', '2026-01-01 15:00:00',
               '2026-01-01 16:00:00', '2026-01-01 17:00:00',
               '2026-01-01 18:00:00', '2026-01-01 19:00:00',
               '2026-01-01 20:00:00', '2026-01-01 21:00:00',
               '2026-01-01 22:00:00', '2026-01-01 23:00:00'],
              dtype='datetime64[us]', name='snapshot', freq='h')

In [3]:
base_mva = 10.0
base_kv = 12.66
z_base = (base_kv**2) / base_mva

Add the network buses

In [4]:
network.add("Bus", "node_1", v_nom=base_kv, generator="substation", carrier="AC")
for i in range (2, 34):
    network.add("Bus", f"node_{i}", v_nom=base_kv, carrier="AC")

network.buses

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
Bus,,,,,,,,,,,,,
node_1,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,substation,
node_2,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
node_3,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
node_4,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
node_5,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
node_6,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
node_7,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
node_8,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
node_9,12.66,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,


Add the external grid at the first node

In [5]:
network.add(
    "Generator", 
    "substation", 
    bus="node_1",
    control="Slack",
    marginal_cost=50,  #dummy cost
    p_nom_extendable=True,
    p_nom_max=5.0,
)
network.generators

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_min_pu,p_max_pu,...,min_up_time,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt
Generator,,,,,,,,,,,,,,,,,,,,,
substation,node_1,Slack,,0.0,0.0,True,0.0,5.0,0.0,1.0,...,0,0,1,0,NaN,NaN,1.0,1.0,1.0,0.0


Create the line data: (from_node, to_node, r(resistance-ohm), x(reactance-ohm) )
And add it to the network
s_nom_max = 10

In [6]:

lines = [
    (1, 2, 0.0922, 0.047),      (2, 3, 0.493, 0.2511),      (3, 4, 0.366, 0.1864), 
    (4, 5, 0.3811, 0.1941),     (5, 6, 0.819, 0.707),       (6, 7, 0.1872, 0.6188), 
    (7, 8, 0.7114, 0.2351),     (8, 9, 1.03, 0.74),         (9, 10, 1.044, 0.74),   
    (10, 11, 0.1966, 0.065),    (11, 12, 0.3744, 0.198),    (12, 13, 1.468, 1.155),
    (13, 14, 0.5416, 0.7129),   (14, 15, 0.591, 0.526),     (15, 16, 0.7463, 0.545),
    (16, 17, 1.289, 1.721),     (17, 18, 0.732, 0.574),     (2, 19, 0.164, 0.1565),
    (19, 20, 1.5042, 1.3554),   (20, 21, 0.4095, 0.4784),   (21, 22, 0.7089, 0.9373),   
    (3, 23, 0.4512, 0.3083),    (23, 24, 0.898, 0.7091),    (24, 25, 0.896, 0.7011), 
    (6, 26, 0.203, 0.1034),     (26, 27, 0.2842, 0.1447),   (27, 28, 1.059, 0.9337), 
    (28, 29, 0.8042, 0.7006),   (29, 30, 0.5075, 0.2585),   (30, 31, 0.9744, 0.963), 
    (31, 32, 0.3105, 0.3619),   (32, 33, 0.341, 0.5302),
]

for i, (from_node, to_node, r, x) in enumerate(lines):
    network.add("Line",
                f"line_{from_node}_{to_node}",
                bus0=f"node_{from_node}",
                bus1=f"node_{to_node}",
                r = r / base_kv,
                x = x / base_kv,
                s_nom = base_mva,
                carrier="AC"
                )
network.lines

,bus0,bus1,type,x,r,g,b,s_nom,s_nom_mod,s_nom_extendable,...,v_ang_min,v_ang_max,sub_network,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt
Line,,,,,,,,,,,,,,,,,,,,,
line_1_2,node_1,node_2,,0.003712,0.007283,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0
line_2_3,node_2,node_3,,0.019834,0.038942,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0
line_3_4,node_3,node_4,,0.014724,0.028910,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0
line_4_5,node_4,node_5,,0.015332,0.030103,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0
line_5_6,node_5,node_6,,0.055845,0.064692,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0
line_6_7,node_6,node_7,,0.048878,0.014787,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0
line_7_8,node_7,node_8,,0.018570,0.056193,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0
line_8_9,node_8,node_9,,0.058452,0.081359,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0
line_9_10,node_9,node_10,,0.058452,0.082464,0.0,0.0,10.0,0.0,False,...,-inf,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Define the nominal loads at the receiving bus 
(bus number, P(kW), Q(kVar) )

In [7]:
loads = {
    2: (100, 60),       3: (90, 40),        4: (120, 80),
    5: (60, 30),        6: (60, 20),        7: (200, 100),
    8: (200, 100),      9: (60, 20),        10: (60, 20),
    11: (45, 30),       12: (60, 35),       13: (60, 35),
    14: (120, 80),      15: (60, 10),       16: (60, 20),
    17: (60, 20),       18: (90, 40),       19: (90, 40),
    20: (90, 40),       21: (90, 40),       22: (90, 40),
    23: (90, 50),       24: (420, 200),     25: (420, 200),
    26: (60, 25),       27: (60, 25),       28: (60, 20),
    29: (120, 70),      30: (200, 600),     31: (150, 70),
    32: (210, 100),     33: (60, 40),
}

Dummy solar and load profiles

In [8]:
solar_profile = np.array([
    0.00, 0.00, 0.00, 0.00,
    0.00, 0.00, 0.10, 0.30,
    0.60, 0.80, 0.90, 1.00,
    1.00, 0.90, 0.70, 0.50,
    0.20, 0.05, 0.00, 0.00,
    0.00, 0.00, 0.00, 0.00
])

load_profile = np.array([
    0.40, 0.30, 0.30, 0.40,
    0.50, 0.60, 0.70, 0.80,
    0.70, 0.60, 0.60, 0.70,
    0.80, 0.80, 0.70, 0.70,
    0.80, 1.00, 0.90, 0.80,
    0.70, 0.60, 0.50, 0.40
])


Create the loads, PV and batteries

In [9]:
for i in range(2, 34):
    p_set = loads[i][0] / 1000 #kW to MW
    q_set = loads[i][1] / 1000 #kVar to MVar

    # add load
    network.add(
        "Load",
        f"load_node_{i}",
        bus=f"node_{i}",
        p_set=load_profile * p_set,
        q_set=load_profile * q_set
    )

    # add extendable PV generator
    network.add(
        "Generator",
        f"PV_node_{i}",
        bus=f"node_{i}",
        p_nom_extendable=True,
        p_max_pu=solar_profile,
        capital_cost=0.0,
        marginal_cost=0.0,
        # carrier="solar"
    )

    # add extendable battery storage system
    network.add(
        "StorageUnit",
        f"battery_node_{i}",
        bus=f"node_{i}",
        capital_cost = 0.0,
        efficiency_store=0.92,
        efficiency_dispatch=0.92,
        max_hours=24.0
    )

Run linear optimal power flow

In [10]:
network.optimize()

Index(['line_1_2', 'line_2_3', 'line_3_4', 'line_4_5', 'line_5_6', 'line_6_7',
       'line_7_8', 'line_8_9', 'line_9_10', 'line_10_11', 'line_11_12',
       'line_12_13', 'line_13_14', 'line_14_15', 'line_15_16', 'line_16_17',
       'line_17_18', 'line_2_19', 'line_19_20', 'line_20_21', 'line_21_22',
       'line_3_23', 'line_23_24', 'line_24_25', 'line_6_26', 'line_26_27',
       'line_27_28', 'line_28_29', 'line_29_30', 'line_30_31', 'line_31_32',
       'line_32_33'],
      dtype='str', name='Line')
Index(['node_1', 'node_2', 'node_3', 'node_4', 'node_5', 'node_6', 'node_7',
       'node_8', 'node_9', 'node_10', 'node_11', 'node_12', 'node_13',
       'node_14', 'node_15', 'node_16', 'node_17', 'node_18', 'node_19',
       'node_20', 'node_21', 'node_22', 'node_23', 'node_24', 'node_25',
       'node_26', 'node_27', 'node_28', 'node_29', 'node_30', 'node_31',
       'node_32', 'node_33'],
      dtype='str', name='Bus')
INFO:linopy.model: Solve problem using Highs solver
INFO:linop

('ok', 'optimal')

In [11]:
network.generators_t.p.substation

snapshot
2026-01-01 00:00:00    1.4860
2026-01-01 01:00:00    1.1145
2026-01-01 02:00:00    1.1145
2026-01-01 03:00:00    1.4860
2026-01-01 04:00:00    1.8575
2026-01-01 05:00:00    2.2290
2026-01-01 06:00:00   -0.0000
2026-01-01 07:00:00   -0.0000
2026-01-01 08:00:00   -0.0000
2026-01-01 09:00:00   -0.0000
2026-01-01 10:00:00   -0.0000
2026-01-01 11:00:00   -0.0000
2026-01-01 12:00:00   -0.0000
2026-01-01 13:00:00   -0.0000
2026-01-01 14:00:00   -0.0000
2026-01-01 15:00:00   -0.0000
2026-01-01 16:00:00   -0.0000
2026-01-01 17:00:00   -0.0000
2026-01-01 18:00:00    3.3435
2026-01-01 19:00:00    2.9720
2026-01-01 20:00:00    2.6005
2026-01-01 21:00:00    2.2290
2026-01-01 22:00:00    1.8575
2026-01-01 23:00:00    1.4860
Freq: h, Name: substation, dtype: float64

In [12]:
network.generators_t.p

Generator,substation,PV_node_2,PV_node_3,PV_node_4,PV_node_5,PV_node_6,PV_node_7,PV_node_8,PV_node_9,PV_node_10,...,PV_node_24,PV_node_25,PV_node_26,PV_node_27,PV_node_28,PV_node_29,PV_node_30,PV_node_31,PV_node_32,PV_node_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2026-01-01 00:00:00,1.4860,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0000,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2026-01-01 01:00:00,1.1145,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0000,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2026-01-01 02:00:00,1.1145,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0000,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2026-01-01 03:00:00,1.4860,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0000,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2026-01-01 04:00:00,1.8575,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0000,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2026-01-01 05:00:00,2.2290,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0000,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2026-01-01 06:00:00,-0.0000,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,2.6005,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2026-01-01 07:00:00,-0.0000,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,2.9720,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2026-01-01 08:00:00,-0.0000,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,2.6005,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0


In [13]:
network.storage_units_t.p

StorageUnit,battery_node_2,battery_node_3,battery_node_4,battery_node_5,battery_node_6,battery_node_7,battery_node_8,battery_node_9,battery_node_10,battery_node_11,...,battery_node_24,battery_node_25,battery_node_26,battery_node_27,battery_node_28,battery_node_29,battery_node_30,battery_node_31,battery_node_32,battery_node_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2026-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 03:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 04:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 05:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 06:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 07:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 08:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
